# Optimize a non-Cartesian MRI trajectory (SNOPY-style)

**Place in the learning path:** this is an advanced nested optimization problem.
The inner problem reconstructs the image $x$; the outer problem updates acquisition
parameters $\theta$. See [`demo_mr_physics.ipynb`](demo_mr_physics.ipynb) for the
forward/inverse foundation and [`demo_mri.ipynb`](demo_mri.ipynb) for NUFFT models.

This notebook builds a small, complete version of the acquisition-learning loop
in [SNOPY](https://arxiv.org/abs/2209.11030): a sampling parameterization produces
a trajectory, `NuSense` simulates its measurements, a differentiable iterative
solver reconstructs an image, and image error updates the sampling parameters.

**Learning goals**

- parameterize radial spokes by learnable rotation angles;
- verify the NUFFT adjoint and trajectory gradient numerically;
- differentiate through a truncated penalized least-squares reconstruction; and
- check whether the learned trajectory improves images that were not used for
  optimization.

The example is deliberately small enough for CUDA, Apple Metal, or CPU. It follows
SNOPY's stack-of-stars angle experiment in two dimensions; it is not a scanner-ready
gradient waveform design.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch

from mirtorch.alg import CG
from mirtorch.linear import Diff2dgram, Identity, NuSense


def mps_supports_complex_fft():
    backend = getattr(torch.backends, "mps", None)
    if backend is None or not backend.is_available():
        return False
    try:
        probe = torch.ones(2, dtype=torch.complex64, device="mps")
        torch.fft.fft(probe)
        torch.mps.synchronize()
        return True
    except (RuntimeError, NotImplementedError):
        return False


def best_available_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if mps_supports_complex_fft():
        return torch.device("mps")
    return torch.device("cpu")


device = best_available_device()
torch.manual_seed(7)
print(f"PyTorch {torch.__version__} | device: {device}")

## 1. The differentiable acquisition problem

For trajectory parameters $\theta$, let $\omega(\theta)$ be the sampled k-space locations and let $A_{\omega}$ be the multi-coil NUFFT. We optimize reconstruction quality through

$$y_{\theta}=A_{\omega(\theta)}x, \qquad \hat{x}_{\theta}=\operatorname{CG}_K\!\left(A_{\omega}^H A_{\omega}+\lambda D^H D+\epsilon I,\; A_{\omega}^H y_{\theta}\right).$$

The training loss combines relative $\ell_1$ and squared $\ell_2$ image error, as in SNOPY. Measurements are regenerated at the current trajectory, so both the simulated acquisition and reconstruction depend on $\theta$; simulated noise is set to zero for a deterministic tutorial.

MIRTorch evaluates the trajectory vector-Jacobian product using the coordinate-weighted NUFFT identity from [Wang and Fessler (2023)](https://arxiv.org/abs/2111.02912). The solver uses `backward_mode="unrolled"`: the default implicit CG backward differentiates its right-hand side, but does not include changes in the operator $A_{\omega}^H A_{\omega}$. Here $K$ is kept small so the complete truncated-reconstruction gradient fits comfortably in memory.

## 2. Make a tiny training distribution

The six training phantoms vary lesion position and orientation. Two held-out variants are used only after optimization. Smooth, root-sum-of-squares-normalized coil maps make the forward model genuinely multi-coil without requiring a download.

In [ ]:
IMAGE_SIZE = 28
NCOILS = 3
NSPOKES = 10
NREADOUT = IMAGE_SIZE

axis = torch.linspace(-1, 1, IMAGE_SIZE, device=device)
yy, xx = torch.meshgrid(axis, axis, indexing="ij")


def ellipse(cx, cy, rx, ry, angle=0.0):
    cosine, sine = math.cos(angle), math.sin(angle)
    x_rotated = cosine * (xx - cx) + sine * (yy - cy)
    y_rotated = -sine * (xx - cx) + cosine * (yy - cy)
    return (x_rotated.square() / rx**2 + y_rotated.square() / ry**2 <= 1).float()


def make_phantom(variant):
    image = 0.90 * ellipse(0.00, 0.00, 0.58, 0.76)
    image += 0.32 * ellipse(-0.22 + 0.025 * variant, 0.04, 0.16, 0.28, 0.25)
    image += 0.48 * ellipse(0.25, -0.18 + 0.02 * variant, 0.11, 0.14, -0.20)
    image -= 0.22 * ellipse(0.04, 0.28, 0.12, 0.09)
    image += 0.12 * ellipse(-0.10, -0.30, 0.24, 0.08, 0.08 * variant)
    return image.clamp_min(0)


train_images = torch.stack([make_phantom(v) for v in (-2, -1, 0, 1, 2, 3)])
heldout_images = torch.stack([make_phantom(v) for v in (-3, 4)])
train_images = train_images.unsqueeze(1).to(torch.complex64)
heldout_images = heldout_images.unsqueeze(1).to(torch.complex64)

coil_centers = ((0.0, -1.2), (1.04, 0.6), (-1.04, 0.6))
coil_maps = []
for center_y, center_x in coil_centers:
    magnitude = (
        torch.exp(
            -((xx - center_x).square() + (yy - center_y).square()) / (2 * 1.25**2)
        )
        + 0.12
    )
    phase = torch.atan2(yy - center_y, xx - center_x)
    coil_maps.append(magnitude * torch.exp(1j * phase))
coil_maps = torch.stack(coil_maps).to(torch.complex64)
coil_maps /= torch.sqrt(coil_maps.abs().square().sum(0, keepdim=True))

fig, axes = plt.subplots(1, 4, figsize=(10, 2.6), constrained_layout=True)
axes[0].imshow(train_images[0, 0].abs().cpu(), cmap="gray")
axes[0].set_title("Training example")
axes[1].imshow(heldout_images[0, 0].abs().cpu(), cmap="gray")
axes[1].set_title("Held-out example")
for index in range(2):
    axes[index + 2].imshow(coil_maps[index].abs().cpu(), cmap="viridis")
    axes[index + 2].set_title(f"Coil {index + 1} magnitude")
for current_axis in axes:
    current_axis.axis("off")
plt.show()

## 3. Parameterize radial rotation angles

A spoke with angle $\theta_s$ samples

$$\omega_s(q)=q[\sin(\theta_s),\cos(\theta_s)], \qquad q\in[-\pi,\pi],$$

in MIRTorch's `(ky, kx)` coordinate order, measured in radians/voxel. Optimizing angles preserves each spoke's center crossing, readout extent, and gradient/slew magnitude. The intentionally compressed initialization leaves a large angular gap, making the learning signal visible in a short example.

In [ ]:
radius = torch.linspace(-math.pi, math.pi, NREADOUT, device=device)
initial_angles = torch.linspace(0, 0.58 * math.pi, NSPOKES, device=device)
initial_angles += 0.03 * torch.randn(NSPOKES, device=device)


def radial_trajectory(angles):
    ky = torch.sin(angles)[:, None] * radius
    kx = torch.cos(angles)[:, None] * radius
    return torch.stack((ky.reshape(-1), kx.reshape(-1))).unsqueeze(0)


def plot_trajectory(current_axis, trajectory, title):
    samples = trajectory.detach().cpu()[0].reshape(2, NSPOKES, NREADOUT)
    for shot in range(NSPOKES):
        current_axis.plot(samples[1, shot], samples[0, shot], linewidth=1.1)
    current_axis.set(
        title=title, xlabel=r"$k_x$ (rad/voxel)", ylabel=r"$k_y$ (rad/voxel)"
    )
    current_axis.set_aspect("equal")
    current_axis.set_xlim(-math.pi, math.pi)
    current_axis.set_ylim(-math.pi, math.pi)
    current_axis.grid(alpha=0.2)


fig, axis = plt.subplots(figsize=(4, 4), constrained_layout=True)
plot_trajectory(axis, radial_trajectory(initial_angles), "Initial trajectory")
plt.show()

## 4. Reconstruct and define the training loss

Each mini-batch gets its own expanded coil-map batch while sharing one trajectory. Five unrolled CG steps solve a roughness-penalized system. A tiny ridge makes the truncated system numerically robust. `backend="auto"` selects cuFINUFFT on CUDA when installed, FINUFFT on supported Linux CPUs, and torchkbnufft otherwise.

In [ ]:
CG_ITERATIONS = 5
ROUGHNESS_WEIGHT = 1e-3
RIDGE_WEIGHT = 1e-5


def reconstruct(images, angles, *, differentiable):
    smaps = coil_maps.unsqueeze(0).expand(images.shape[0], -1, -1, -1)
    operator = NuSense(
        smaps,
        radial_trajectory(angles),
        backend="auto",
        numpoints=6,
        grid_size=2,
    )
    measurements = operator * images
    right_hand_side = operator.H * measurements
    normal = (
        operator.H * operator
        + ROUGHNESS_WEIGHT * Diff2dgram(operator.size_in, compile=False)
        + RIDGE_WEIGHT * Identity(operator.size_in)
    )
    solver = CG(
        normal,
        max_iter=CG_ITERATIONS,
        tol=0,
        backward_mode="unrolled" if differentiable else "implicit",
    )
    reconstruction = solver.run(torch.zeros_like(images), right_hand_side)
    return reconstruction, operator


def quality_loss(reconstruction, reference):
    error = reconstruction - reference
    relative_l1 = error.abs().mean() / reference.abs().mean()
    relative_l2 = error.abs().square().mean() / reference.abs().square().mean()
    return 0.1 * relative_l1 + relative_l2


def image_metrics(reconstruction, reference):
    error_power = (reconstruction - reference).abs().square()
    nrmse = torch.sqrt(error_power.sum() / reference.abs().square().sum())
    psnr = 20 * torch.log10(reference.abs().amax() / torch.sqrt(error_power.mean()))
    return nrmse.item(), psnr.item()

## 5. Check the mathematics before optimizing

Two inexpensive tests catch the most consequential mistakes. First, $\langle Ax,y\rangle=\langle x,A^Hy\rangle$ checks the complex adjoint. Second, the efficient trajectory VJP is compared with ordinary PyTorch autograd through a tiny exact nonuniform DFT. The exact transform is practical only at this teaching scale, but it is a clean numerical oracle because it contains no interpolation or table lookup.

In [ ]:
def relative_l2_error(actual, expected):
    return torch.sqrt(
        (actual - expected).abs().square().sum()
        / expected.abs().square().sum().clamp_min(1e-12)
    )


def direct_nudft_forward(images, smaps, trajectory, grid_size):
    coordinates = torch.arange(
        -(IMAGE_SIZE // 2),
        (IMAGE_SIZE - 1) // 2 + 1,
        dtype=trajectory.dtype,
        device=trajectory.device,
    )
    grid_y, grid_x = torch.meshgrid(coordinates, coordinates, indexing="ij")
    phase = (
        trajectory[:, 0, :, None, None] * grid_y
        + trajectory[:, 1, :, None, None] * grid_x
    )
    coil_images = images * smaps
    transformed = (coil_images.unsqueeze(2) * torch.exp(-1j * phase).unsqueeze(1)).sum(
        dim=(-2, -1)
    )
    return transformed / math.sqrt(math.prod(grid_size))


check_smaps = coil_maps.unsqueeze(0).expand(2, -1, -1, -1)
check_trajectory = radial_trajectory(initial_angles).detach().requires_grad_()
check_operator = NuSense(
    check_smaps,
    check_trajectory,
    backend="auto",
    numpoints=6,
    grid_size=2,
)
probe_image = torch.randn_like(train_images[:2])
probe_samples = torch.randn(
    check_operator.size_out, dtype=torch.complex64, device=device
)
left_inner_product = ((check_operator * probe_image).conj() * probe_samples).sum()
right_inner_product = (probe_image.conj() * (check_operator.H * probe_samples)).sum()
adjoint_error = (left_inner_product - right_inner_product).abs() / torch.maximum(
    left_inner_product.abs(), right_inner_product.abs()
).clamp_min(1e-8)

actual_samples = check_operator * probe_image
upstream = torch.randn_like(actual_samples)
actual_scalar = (upstream.conj() * actual_samples).sum().real
(actual_gradient,) = torch.autograd.grad(actual_scalar, check_trajectory)

exact_trajectory = check_trajectory.detach().clone().requires_grad_()
exact_samples = direct_nudft_forward(
    probe_image, check_smaps, exact_trajectory, check_operator.grid_size
)
exact_scalar = (upstream.conj() * exact_samples).sum().real
(exact_gradient,) = torch.autograd.grad(exact_scalar, exact_trajectory)
forward_error = relative_l2_error(actual_samples.detach(), exact_samples.detach())
gradient_error = relative_l2_error(actual_gradient, exact_gradient)
gradient_cosine = (actual_gradient * exact_gradient).sum() / (
    actual_gradient.square().sum().sqrt() * exact_gradient.square().sum().sqrt()
)

print(f"NUFFT backend: {check_operator.backend}")
print(f"Adjoint relative error: {adjoint_error.item():.2e}")
print(f"NUFFT vs exact NUDFT relative error: {forward_error.item():.2e}")
print(f"Trajectory VJP relative error: {gradient_error.item():.2e}")
print(f"Trajectory VJP cosine similarity: {gradient_cosine.item():.8f}")
assert adjoint_error.item() < 5e-4
assert forward_error.item() < 3e-3
assert gradient_error.item() < 3e-3
assert gradient_cosine.item() > 0.999

## 6. Optimize on mini-batches

Adam keeps this teaching run deterministic; the SNOPY paper also reports that stochastic gradient Langevin dynamics can help escape poor local minima. We cycle through two-image mini-batches and linearly decay the learning rate. The trajectory is rebuilt each step so no stale NUFFT state is reused.

In [ ]:
angles = torch.nn.Parameter(initial_angles.detach().clone())
optimizer = torch.optim.Adam([angles], lr=0.035)
OPTIMIZATION_STEPS = 30
BATCH_SIZE = 2
loss_history = []
gradient_history = []
start_time = time.perf_counter()

for step in range(OPTIMIZATION_STEPS):
    first = (BATCH_SIZE * step) % len(train_images)
    indices = torch.tensor(
        [(first + offset) % len(train_images) for offset in range(BATCH_SIZE)],
        device=device,
    )
    batch = train_images.index_select(0, indices)

    optimizer.zero_grad(set_to_none=True)
    reconstruction, _ = reconstruct(batch, angles, differentiable=True)
    loss = quality_loss(reconstruction, batch)
    if not torch.isfinite(loss):
        raise RuntimeError("The trajectory objective became non-finite")
    loss.backward()
    if not torch.isfinite(angles.grad).all():
        raise RuntimeError("The trajectory gradient became non-finite")
    gradient_norm = torch.nn.utils.clip_grad_norm_([angles], max_norm=1.0)
    optimizer.param_groups[0]["lr"] = 0.035 * (1 - step / OPTIMIZATION_STEPS)
    optimizer.step()

    loss_history.append(loss.detach().item())
    gradient_history.append(torch.as_tensor(gradient_norm).detach().item())
    if step % 5 == 0 or step == OPTIMIZATION_STEPS - 1:
        print(
            f"step {step:02d} | loss {loss_history[-1]:.4f} | "
            f"|grad| {gradient_history[-1]:.3e}"
        )

elapsed = time.perf_counter() - start_time
optimized_angles = angles.detach().clone()
print(f"Optimization time: {elapsed:.1f} s on {device}")
assert min(loss_history[-5:]) < 0.9 * loss_history[0]

## 7. Evaluate held-out images

The comparison below never contributed to an update. A useful learned pattern should reduce held-out NRMSE and improve PSNR, not merely lower the mini-batch objective. The angle gaps provide a second, geometric explanation: optimization should fill the large missing angular sector.

In [ ]:
with torch.no_grad():
    initial_reconstruction, _ = reconstruct(
        heldout_images, initial_angles, differentiable=False
    )
    optimized_reconstruction, _ = reconstruct(
        heldout_images, optimized_angles, differentiable=False
    )

initial_nrmse, initial_psnr = image_metrics(initial_reconstruction, heldout_images)
optimized_nrmse, optimized_psnr = image_metrics(
    optimized_reconstruction, heldout_images
)
print(f"{'trajectory':<12} {'NRMSE':>10} {'PSNR (dB)':>12}")
print(f"{'initial':<12} {initial_nrmse:>10.4f} {initial_psnr:>12.2f}")
print(f"{'optimized':<12} {optimized_nrmse:>10.4f} {optimized_psnr:>12.2f}")


def angular_gaps_degrees(current_angles):
    wrapped = torch.remainder(current_angles, math.pi).sort().values
    closed = torch.cat((wrapped, wrapped[:1] + math.pi))
    return torch.diff(closed) * 180 / math.pi


initial_gaps = angular_gaps_degrees(initial_angles).cpu()
optimized_gaps = angular_gaps_degrees(optimized_angles).cpu()
print(f"Largest angle gap: {initial_gaps.max():.1f}° → {optimized_gaps.max():.1f}°")

fig, axes = plt.subplots(2, 3, figsize=(11, 7), constrained_layout=True)
plot_trajectory(axes[0, 0], radial_trajectory(initial_angles), "Initial trajectory")
plot_trajectory(axes[0, 1], radial_trajectory(optimized_angles), "Optimized trajectory")
axes[0, 2].plot(loss_history, color="tab:blue")
axes[0, 2].set(title="Mini-batch objective", xlabel="Update", ylabel="Loss")
axes[0, 2].grid(alpha=0.25)

display_max = heldout_images[0, 0].abs().max().cpu()
images = (
    ("Held-out reference", heldout_images[0, 0]),
    (f"Initial\nmean {initial_psnr:.1f} dB", initial_reconstruction[0, 0]),
    (f"Optimized\nmean {optimized_psnr:.1f} dB", optimized_reconstruction[0, 0]),
)
for current_axis, (title, image) in zip(axes[1], images):
    current_axis.imshow(image.abs().cpu(), cmap="gray", vmin=0, vmax=display_max)
    current_axis.set_title(title)
    current_axis.axis("off")
plt.show()

assert optimized_nrmse < initial_nrmse
assert optimized_psnr > initial_psnr
assert optimized_gaps.max() < initial_gaps.max()

## Takeaway and extensions

This run improves reconstruction by redistributing radial angles while retaining a familiar acquisition family. It exercises the same essential path as SNOPY: **sampling parameters → NUFFT → iterative reconstruction → image loss → trajectory update**. The result is an educational reproduction, not the paper's full 3D experiment.

For freeform waveform optimization, replace the angle model by $\omega=Bc$ using quadratic B-spline coefficients and add hinge penalties above scanner gradient and slew limits, as described in SNOPY. A deployable sequence also needs prewinder/rewinder design, gradient-delay and eddy-current checks, and validation on the target scanner. Larger 3D problems benefit from cuFINUFFT, Toeplitz normal operators, and a memory-efficient implicit inverse VJP rather than the unrolled CG used here. `Gmri` can replace `NuSense` when the training distribution should include B0 maps and readout times.

**References:** [SNOPY paper](https://arxiv.org/abs/2209.11030) · [official SNOPY repository](https://github.com/guanhuaw/SNOPY) · [efficient NUFFT Jacobian paper](https://arxiv.org/abs/2111.02912)